To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News

**NEW** Unsloth now supports training the new **gpt-oss** model from OpenAI! You can start finetune gpt-oss for free with our **[Colab notebook](https://x.com/UnslothAI/status/1953896997867729075)**!

Unsloth now supports Text-to-Speech (TTS) models. Read our [guide here](https://docs.unsloth.ai/basics/text-to-speech-tts-fine-tuning).

Read our **[Gemma 3N Guide](https://docs.unsloth.ai/basics/gemma-3n-how-to-run-and-fine-tune)** and check out our new **[Dynamic 2.0](https://docs.unsloth.ai/basics/unsloth-dynamic-2.0-ggufs)** quants which outperforms other quantization methods!

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.55.4

### Unsloth

In [ ]:
import unsloth
from unsloth import FastLanguageModel
import torch
max_seq_length = 8192 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    "unsloth/llama-2-7b-bnb-4bit",
    "unsloth/llama-2-13b-bnb-4bit",
    "unsloth/codellama-34b-bnb-4bit",
    "unsloth/tinyllama-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit", # New Google 6 trillion tokens model 2.5x faster!
    "unsloth/gemma-2b-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-v0.3", # Choose ANY! eg teknium/OpenHermes-2.5-Mistral-7B
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

==((====))==  Unsloth 2025.9.2: Fast Mistral patching. Transformers: 4.55.4.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
#abrir ventana para cargar dataset
from google.colab import files
uploaded = files.upload()

Saving legal_dataset.json to legal_dataset.json


In [ ]:
#cargar dataset
import json

with open("/content/legal_dataset.json", "r", encoding="utf-8-sig") as f:
    data = json.load(f)

print(f"Se cargaron {len(data)} ejemplos.")

def split_long_texts(dataset, tokenizer, max_tokens):
    new_dataset = []
    for sample in dataset:
        tokens = tokenizer.encode(sample["text"])
        for i in range(0, len(tokens), max_tokens):
            chunk_tokens = tokens[i:i+max_tokens]
            chunk_text = tokenizer.decode(chunk_tokens)
            new_dataset.append({"text": chunk_text})
    return new_dataset

Se cargaron 196 ejemplos.


In [ ]:
#PARA VER SI ESTAN TODA LAS SECCIONES
def check_sections(data):
    for i, ex in enumerate(data):
        out = ex["output"]
        for sec in ["ANTECEDENTES", "BASE LEGAL", "ANÁLISIS", "CONCLUSIÓN", "RECOMENDACIONES"]:
            if sec not in out:
                print(f"Ejemplo {i} le falta: {sec}")

check_sections(data)


Ejemplo 6 le falta: BASE LEGAL
Ejemplo 11 le falta: RECOMENDACIONES
Ejemplo 12 le falta: BASE LEGAL
Ejemplo 12 le falta: RECOMENDACIONES
Ejemplo 16 le falta: BASE LEGAL
Ejemplo 16 le falta: RECOMENDACIONES
Ejemplo 22 le falta: BASE LEGAL
Ejemplo 22 le falta: RECOMENDACIONES
Ejemplo 23 le falta: BASE LEGAL
Ejemplo 26 le falta: BASE LEGAL
Ejemplo 26 le falta: ANÁLISIS
Ejemplo 27 le falta: BASE LEGAL
Ejemplo 27 le falta: ANÁLISIS
Ejemplo 31 le falta: BASE LEGAL
Ejemplo 31 le falta: ANÁLISIS
Ejemplo 32 le falta: ANÁLISIS
Ejemplo 34 le falta: ANTECEDENTES
Ejemplo 34 le falta: ANÁLISIS
Ejemplo 41 le falta: BASE LEGAL
Ejemplo 42 le falta: RECOMENDACIONES
Ejemplo 47 le falta: RECOMENDACIONES
Ejemplo 49 le falta: CONCLUSIÓN
Ejemplo 50 le falta: BASE LEGAL
Ejemplo 57 le falta: ANÁLISIS
Ejemplo 60 le falta: BASE LEGAL
Ejemplo 60 le falta: RECOMENDACIONES
Ejemplo 61 le falta: BASE LEGAL
Ejemplo 65 le falta: RECOMENDACIONES
Ejemplo 70 le falta: BASE LEGAL
Ejemplo 70 le falta: RECOMENDACIONES
Ejempl

In [ ]:
#PARA VER NUMERO DE TOKENS

from transformers import AutoTokenizer
import json
import numpy as np

# ⚡ Usa el mismo tokenizer que cargas con FastLanguageModel
# Si ya tienes "tokenizer" en memoria puedes usarlo directamente
# tokenizer = AutoTokenizer.from_pretrained("unsloth/Meta-Llama-3.1-8B")

# === 1. Cargar dataset en formato JSON ===
with open("legal_dataset.json", "r", encoding="utf-8-sig") as f:
    data = json.load(f)

# === 2. Calcular longitud en tokens por cada muestra ===
lengths = []
for i, sample in enumerate(data):
    # Aquí depende de cómo está tu dataset (alpaca style)
    instr = sample.get("instruction", "")
    inp   = sample.get("input", "")
    out   = sample.get("output", "")
    text  = f"{instr}\n{inp}\n{out}"

    tokens = tokenizer.encode(text)
    lengths.append(len(tokens))

# === 3. Estadísticas ===
print("📊 Estadísticas de longitud en tokens:")
print(f" - Mínimo: {np.min(lengths)}")
print(f" - Promedio: {np.mean(lengths):.2f}")
print(f" - Máximo: {np.max(lengths)}")

# === 4. Revisar cuántos ejemplos se pasan de 8192 ===
over_limit = [i for i, l in enumerate(lengths) if l > 8192]
print(f"⚠️ Ejemplos que superan 8192 tokens: {len(over_limit)}")

if over_limit:
    print("IDs de ejemplos largos:", over_limit[:10])  # muestra los primeros 10

📊 Estadísticas de longitud en tokens:
 - Mínimo: 1935
 - Promedio: 3994.94
 - Máximo: 8183
⚠️ Ejemplos que superan 8192 tokens: 0


<a name="Data"></a>
### Data Prep
We now use the Alpaca dataset from [yahma](https://huggingface.co/datasets/yahma/alpaca-cleaned), which is a filtered version of 52K of the original [Alpaca dataset](https://crfm.stanford.edu/2023/03/13/alpaca.html). You can replace this code section with your own data prep.

**[NOTE]** To train only on completions (ignoring the user's input) read TRL's docs [here](https://huggingface.co/docs/trl/sft_trainer#train-on-completions-only).

**[NOTE]** Remember to add the **EOS_TOKEN** to the tokenized output!! Otherwise you'll get infinite generations!

If you want to use the `llama-3` template for ShareGPT datasets, try our conversational [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Mistral_v0.3_(7B)-Conversational.ipynb)

For text completions like novel writing, try this [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Mistral_(7B)-Text_Completion.ipynb).

In [ ]:
# ========================================================
# 1. IMPORTAR Y CONFIGURAR
# ========================================================
import json
from datasets import Dataset

# Suponiendo que ya tienes cargado tu tokenizer de Unsloth
# tokenizer = ...  (FastLanguageModel.from_pretrained(...))

# ========================================================
# 2. DEFINIR PROMPT FORMATO ALPACA CON LA ESTRUCTURA DESEADA
# ========================================================
alpaca_prompt = """Redacta un informe legal siguiendo estrictamente la estructura:

1. ANTECEDENTES
2. BASE LEGAL
3. ANÁLISIS
4. CONCLUSIONES
5. RECOMENDACIONES

Asegúrate de **no terminar el texto sin incluir todas las secciones**.
Finaliza siempre con RECOMENDACIONES.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # añadimos EOS para evitar que la generación sea infinita
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text": texts }

# ========================================================
# 3. DEFINIR FUNCIÓN DE TROCEO (CHUNKING)
# ========================================================
def chunk_dataset(dataset, tokenizer, max_tokens=4096, overlap=200):
    """
    dataset: lista de dicts con keys instruction, input, output
    max_tokens: máximo de tokens que debe tener cada bloque
    overlap: tokens de solapamiento entre bloques consecutivos
    """
    new_dataset = []
    for sample in dataset:
        instr = sample.get("instruction", "")
        inp   = sample.get("input", "")
        out   = sample.get("output", "")
        tokens = tokenizer.encode(out)

        start = 0
        while start < len(tokens):
            end = start + max_tokens
            chunk_tokens = tokens[start:end]
            chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)

            new_dataset.append({
                "instruction": instr,
                "input": inp,
                "output": chunk_text
            })

            # avanzamos con solapamiento
            start = end - overlap if end - overlap > 0 else end
    return new_dataset

# ========================================================
# 4. CARGAR EL JSON ORIGINAL Y TROCEAR
# ========================================================
with open("legal_dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Dataset original: {len(data)} ejemplos")

# Elige max_tokens ≈ max_seq_length del modelo
data_chunked = chunk_dataset(data, tokenizer, max_tokens=4096, overlap=200)

print(f"Después de chunking: {len(data_chunked)} ejemplos")

# ========================================================
# 5. CREAR DATASET HUGGINGFACE Y MAPEAR PROMPTS
# ========================================================
dataset = Dataset.from_list(data_chunked)
dataset = dataset.map(formatting_prompts_func, batched=True)

# Ahora dataset["text"] contiene cada bloque listo para entrenamiento SFT
# ========================================================
# 6. CONTINUAR CON TU SFTTrainer COMO SIEMPRE
# ========================================================
# from trl import SFTTrainer, SFTConfig
# trainer = SFTTrainer(
#     model = model,
#     tokenizer = tokenizer,
#     train_dataset = dataset,
#     dataset_text_field = "text",
#     max_seq_length = 4096,   # igual al que usaste en chunk_dataset
#     packing = False,
#     args = SFTConfig(...),
# )


Dataset original: 196 ejemplos
Después de chunking: 280 ejemplos


Map:   0%|          | 0/280 [00:00<?, ? examples/s]

In [ ]:
def chunk_dataset(dataset, tokenizer, max_tokens=4096, overlap=200):
    new_dataset = []
    for sample in dataset:
        instr = sample.get("instruction", "")
        inp   = sample.get("input", "")
        out   = sample.get("output", "")
        tokens = tokenizer.encode(out)

        start = 0
        while start < len(tokens):
            end = start + max_tokens
            chunk_tokens = tokens[start:end]
            chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            new_dataset.append({
                "instruction": instr,
                "input": inp,
                "output": chunk_text
            })
            start = end - overlap if end - overlap > 0 else end
    return new_dataset

# 2. Trocear si el output supera max_tokens
data_chunked = chunk_dataset(data, tokenizer, max_tokens=4096, overlap=200)
print(f"Antes: {len(data)} ejemplos. Después de chunking: {len(data_chunked)} ejemplos.")

# 3. Pasar a Dataset HuggingFace
dataset = Dataset.from_list(data_chunked)

# 4. Mapear al formato de entrenamiento con tu alpaca_prompt
EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }
#importar dataset
from datasets import Dataset
from datasets import load_dataset
dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"Antes: {len(data)} ejemplos. Después de chunking: {len(data_chunked)} ejemplos.")

'''alpaca_prompt = """Redacta un informe legal siguiendo estrictamente la estructura:

1. ANTECEDENTES
2. BASE LEGAL
3. ANÁLISIS
4. CONCLUSIONES
5. RECOMENDACIONES

Asegúrate de **no terminar el texto sin incluir todas las secciones**.
Finaliza siempre con RECOMENDACIONES.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

#importar dataset
from datasets import Dataset
from datasets import load_dataset
dataset = Dataset.from_list(data)
dataset = dataset.map(formatting_prompts_func, batched = True,)'''

Antes: 196 ejemplos. Después de chunking: 280 ejemplos.


NameError: name 'Dataset' is not defined

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

In [ ]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 40, # Set num_train_epochs = 1 for full training runs
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/280 [00:00<?, ? examples/s]

In [ ]:
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# 👉 Dividir dataset en train y eval (10% para validación)
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = split_dataset["train"]
eval_dataset  = split_dataset["test"]

print(f"Train: {len(train_dataset)} ejemplos")
print(f"Eval: {len(eval_dataset)} ejemplos")

# ==============================
# Definir Trainer con eval_dataset
# ==============================
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset  = eval_dataset,          # 👈 Aquí va
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 200,                 # ejemplo
        learning_rate = 2e-4,
        logging_steps = 10,
        eval_strategy="steps",           # 👈 Evalúa cada cierto número de steps
        eval_steps=50,                   # 👈 Cada 50 steps hace evaluación
        save_strategy="steps",           # 👈 Guarda checkpoints
        save_steps=50,
        output_dir="outputs",
        report_to="none",
        seed = 3407,
    ),
)


Train: 252 ejemplos
Eval: 28 ejemplos


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/252 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/28 [00:00<?, ? examples/s]

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.748 GB.
4.52 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 252 | Num Epochs = 7 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)


Step,Training Loss,Validation Loss
50,0.533600,0.543483
100,0.295500,0.470446
150,0.191200,0.462218
200,0.122600,0.474216


Unsloth: Not an error, but MistralForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

540.6501 seconds used for training.
9.01 minutes used for training.
Peak reserved memory = 5.836 GB.
Peak reserved memory for training = 1.316 GB.
Peak reserved memory % of max memory = 39.571 %.
Peak reserved memory for training % of max memory = 8.923 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Continue the fibonnaci sequence.", # instruction
        "1, 1, 2, 3, 5, 8", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

['<s> Redacta un informe legal siguiendo estrictamente la estructura:\n\n1. ANTECEDENTES\n2. BASE LEGAL\n3. ANÁLISIS\n4. CONCLUSIONES\n5. RECOMENDACIONES\n\nAsegúrate de **no terminar el texto sin incluir todas las secciones**.\nFinaliza siempre con RECOMENDACIONES.\n\n### Instruction:\nContinue the fibonnaci sequence.\n\n### Input:\n1, 1, 2, 3, 5, 8\n\n### Response:\n\nMediante el presente me dirijo a Ud. en atención al Proveído dispuesto por su Despacho así como del Informe remitido por el Abog. Abel M. Sánchez, Jefe de la Oficina de Asesor']

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
instruction = """Completa el siguiente informe legal, rellenando cada sección con contenido relevante:

I. ANTECEDENTES
[Escribe aquí los antecedentes]

II. BASE LEGAL
[Escribe aquí la base legal]

III. ANÁLISIS
[Escribe aquí el análisis]

IV. CONCLUSIONES
[Escribe aquí las conclusiones]

V. RECOMENDACIONES
[Escribe aquí las recomendaciones]

No dejes ninguna sección vacía. Finaliza siempre con RECOMENDACIONES.
"""

inputs = tokenizer(
[
    alpaca_prompt.format(
        instruction, # instruction
        "INFORME N°094-2025-UEI/OPP/MPC-BPR\nA        : Lic. Edder Joshimar Warthon Gamarra Jefe de la Oficina de Planificación y Presupuesto DE      : Brady Palma Rodríguez Encargado de la Unidad de Estadística e Informática ASUNTO : Solicitud de informe legal para aprobación del Plan de Gobierno Digital REFERENCIA : Plan de Gobierno Digital – Municipalidad Provincial de Calca, versión preliminar 2025 FECHA : Calca, 02 de agosto del 2025\nMe dirijo a usted para saludarlo cordialmente y, a la vez, informar que la Unidad de Estadística e Informática ha culminado la elaboración del documento preliminar del Plan de Gobierno Digital de la Municipalidad Provincial de Calca, correspondiente al periodo 2025–2027, en el marco del cumplimiento de la Política Nacional de Transformación Digital y las disposiciones emitidas por la Secretaría de Gobierno y Transformación Digital de la PCM.\nANTECEDENTES\nLa implementación del Plan de Gobierno Digital es una acción estratégica que responde a los lineamientos establecidos por el Decreto Supremo N.º 029-2021-PCM y la Ley N.º 1412, Ley de Gobierno Digital.\nEn este sentido, se ha trabajado un documento técnico que contiene los objetivos, líneas de acción, metas, responsables y cronograma de actividades, que busca fortalecer el ecosistema digital institucional, garantizar la interoperabilidad, impulsar la identidad y servicios digitales, y promover la seguridad digital.\nCabe señalar que la aprobación del presente plan requiere contar previamente con el informe legal correspondiente, el cual determine la viabilidad normativa y sujeción a las disposiciones vigentes.\nANÁLISIS\nLa emisión del informe legal permitirá sustentar formal y legalmente la aprobación del Plan de Gobierno Digital, a fin de remitirlo posteriormente al Pleno del Concejo Municipal, y así cumplir con su implementación de acuerdo con la normativa nacional.\nEste procedimiento es imprescindible para garantizar la legalidad de las acciones institucionales en el marco del proceso de transformación digital en la Municipalidad Provincial de Calca.\nCONCLUSIÓN\nPor lo expuesto, se solicita de manera formal la emisión del informe legal correspondiente, a fin de continuar con el procedimiento de aprobación del Plan de Gobierno Digital de la Municipalidad Provincial de Calca.\nSin otro particular, quedo atento a cualquier consulta o documentación adicional que se requiera para el trámite correspondiente.", # input
        "" # output vacío
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
output = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=1500,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.05

)

decoded = tokenizer.decode(output[0], skip_special_tokens=True)
print("\n\n=== INFORME LEGAL GENERADO ===\n")
print(decoded)
'''
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 500)'''

<s> Redacta un informe legal siguiendo estrictamente la estructura:

1. ANTECEDENTES
2. BASE LEGAL
3. ANÁLISIS
4. CONCLUSIONES
5. RECOMENDACIONES

Asegúrate de **no terminar el texto sin incluir todas las secciones**.
Finaliza siempre con RECOMENDACIONES.

### Instruction:
Completa el siguiente informe legal, rellenando cada sección con contenido relevante:

I. ANTECEDENTES
[Escribe aquí los antecedentes]

II. BASE LEGAL
[Escribe aquí la base legal]

III. ANÁLISIS
[Escribe aquí el análisis]

IV. CONCLUSIONES
[Escribe aquí las conclusiones]

V. RECOMENDACIONES
[Escribe aquí las recomendaciones]

No dejes ninguna sección vacía. Finaliza siempre con RECOMENDACIONES.


### Input:
INFORME N°094-2025-UEI/OPP/MPC-BPR
A        : Lic. Edder Joshimar Warthon Gamarra Jefe de la Oficina de Planificación y Presupuesto DE      : Brady Palma Rodríguez Encargado de la Unidad de Estadística e Informática ASUNTO : Solicitud de informe legal para aprobación del Plan de Gobierno Digital REFERENCIA : Plan 

'\n_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 500)'

In [ ]:
from transformers import TextStreamer

# 1. Activamos inferencia rápida
FastLanguageModel.for_inference(model)

# 2. Prompt con delimitadores claros
instruction = """Redacta un informe legal con la siguiente estructura EXACTA:

I. ANTECEDENTES
II. BASE LEGAL
III. ANÁLISIS
IV. CONCLUSIONES
V. RECOMENDACIONES

⚠️ Cada sección debe aparecer **una sola vez** y en este orden.
No repitas secciones ya escritas.
Debes terminar obligatoriamente en RECOMENDACIONES.
"""

# 3. Tokenizar entrada
inputs = tokenizer(
[
    alpaca_prompt.format(
        instruction,
        "INFORME N°094-2025-UEI/OPP/MPC-BPR\nA        : Lic. Edder Joshimar Warthon Gamarra Jefe de la Oficina de Planificación y Presupuesto DE      : Brady Palma Rodríguez Encargado de la Unidad de Estadística e Informática ASUNTO : Solicitud de informe legal para aprobación del Plan de Gobierno Digital REFERENCIA : Plan de Gobierno Digital – Municipalidad Provincial de Calca, versión preliminar 2025 FECHA : Calca, 02 de agosto del 2025\nMe dirijo a usted para saludarlo cordialmente y, a la vez, informar que la Unidad de Estadística e Informática ha culminado la elaboración del documento preliminar del Plan de Gobierno Digital de la Municipalidad Provincial de Calca, correspondiente al periodo 2025–2027, en el marco del cumplimiento de la Política Nacional de Transformación Digital y las disposiciones emitidas por la Secretaría de Gobierno y Transformación Digital de la PCM.\nANTECEDENTES\nLa implementación del Plan de Gobierno Digital es una acción estratégica que responde a los lineamientos establecidos por el Decreto Supremo N.º 029-2021-PCM y la Ley N.º 1412, Ley de Gobierno Digital.\nEn este sentido, se ha trabajado un documento técnico que contiene los objetivos, líneas de acción, metas, responsables y cronograma de actividades, que busca fortalecer el ecosistema digital institucional, garantizar la interoperabilidad, impulsar la identidad y servicios digitales, y promover la seguridad digital.\nCabe señalar que la aprobación del presente plan requiere contar previamente con el informe legal correspondiente, el cual determine la viabilidad normativa y sujeción a las disposiciones vigentes.\nANÁLISIS\nLa emisión del informe legal permitirá sustentar formal y legalmente la aprobación del Plan de Gobierno Digital, a fin de remitirlo posteriormente al Pleno del Concejo Municipal, y así cumplir con su implementación de acuerdo con la normativa nacional.\nEste procedimiento es imprescindible para garantizar la legalidad de las acciones institucionales en el marco del proceso de transformación digital en la Municipalidad Provincial de Calca.\nCONCLUSIÓN\nPor lo expuesto, se solicita de manera formal la emisión del informe legal correspondiente, a fin de continuar con el procedimiento de aprobación del Plan de Gobierno Digital de la Municipalidad Provincial de Calca.\nSin otro particular, quedo atento a cualquier consulta o documentación adicional que se requiera para el trámite correspondiente.", # input
        "" # output vacío
    )
], return_tensors="pt").to("cuda")

# 4. Calcular tokens disponibles
prompt_len = inputs["input_ids"].shape[1]
max_seq_length = model.config.max_position_embeddings
espacio_libre = max_seq_length - prompt_len
max_new_tokens = max(500, espacio_libre - 20)

print(f"Prompt tokens: {prompt_len}")
print(f"Espacio libre: {espacio_libre}")
print(f"max_new_tokens ajustado: {max_new_tokens}")

# 5. Generación controlada
text_streamer = TextStreamer(tokenizer)

output = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=max_new_tokens,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.2,   # 🔑 evita repeticiones
    no_repeat_ngram_size=4    # 🔑 no permite repetir bloques de 4 tokens
)

decoded = tokenizer.decode(output[0], skip_special_tokens=True)

# 6. Post-procesamiento: cortar donde termine RECOMENDACIONES
if "RECOMENDACIONES" in decoded:
    decoded = decoded.split("RECOMENDACIONES")[-2] + "RECOMENDACIONES" + decoded.split("RECOMENDACIONES")[-1]

print("\n\n=== INFORME LEGAL GENERADO ===\n")
print(decoded)


Prompt tokens: 947
Espacio libre: 31821
max_new_tokens ajustado: 31801
<s> Redacta un informe legal siguiendo estrictamente la estructura:

1. ANTECEDENTES
2. BASE LEGAL
3. ANÁLISIS
4. CONCLUSIONES
5. RECOMENDACIONES

Asegúrate de **no terminar el texto sin incluir todas las secciones**.
Finaliza siempre con RECOMENDACIONES.

### Instruction:
Redacta un informe legal con la siguiente estructura EXACTA:

I. ANTECEDENTES
II. BASE LEGAL
III. ANÁLISIS
IV. CONCLUSIONES
V. RECOMENDACIONES

⚠️ Cada sección debe aparecer **una sola vez** y en este orden.
No repitas secciones ya escritas.
Debes terminar obligatoriamente en RECOMENDACIONES.


### Input:
INFORME N°094-2025-UEI/OPP/MPC-BPR
A        : Lic. Edder Joshimar Warthon Gamarra Jefe de la Oficina de Planificación y Presupuesto DE      : Brady Palma Rodríguez Encargado de la Unidad de Estadística e Informática ASUNTO : Solicitud de informe legal para aprobación del Plan de Gobierno Digital REFERENCIA : Plan de Gobierno Digital – Municipalid

In [ ]:
#PARA SOLO MOSTRAR RESPONSE
# Extraer solo lo que sigue después de '### Response:'
if "### Response:" in decoded:
    response_only = decoded.split("### Response:")[1].strip()
else:
    response_only = decoded  # fallback si no lo encuentra

print("\n\n=== INFORME LEGAL GENERADO ===\n")
print(response_only)



=== INFORME LEGAL GENERADO ===

Mediante el presente me dirijo a Ud. en atención al Proveído dispuesto por su Despacho así como del Informe remitido por el Encargado Especialista de la Unida d’Estátstica e Informatica de la MPC, Brady Paloma Rodriguez, sobre APROBACION DEL PLAN DE GOVERNAMIENTO DIGITAL - MPC. I. ANTECEDETES: 1.1.- Que, mediante Informe N° 094 - 20-25- UEI / OPP / MPC - BPR de fecha 02.08.2022, el Encargadode la Unidad Estadístico e Informaticade la MunicipalidadProvincial de Calcamediante el Especialista BradyPalomarodríguez, hace de conocimiento que la Municipalidad provincial de calca ha concluido la elaboracióndel Documento Preliminar delPlan de GobernancDigital de la Municipalidaddel Periódol 2015-2017,enmarcado en el cumplimiento del Lineamientoestablecido por elDecretoSupremo N° 29- 20 21- PCM y la Lei N° 14 12,Lei de Gobernamiento Digital; II. BASELEGAL: • ConstituciónPolítica del Estado. • Ley N° 30625, Ley Orgánica de Municipalidades • Decreto Legislativo N°

In [ ]:

#PARA VERIFICAR QUE PARTES FALTAN
expected_sections = ["ANTECEDENTES", "BASE LEGAL", "ANÁLISIS", "CONCLUSIONES", "RECOMENDACIONES"]
missing = [sec for sec in expected_sections if sec not in response_only.upper()]

if missing:
    print("Faltan las siguientes secciones:", missing)
else:
  print("completo")

Faltan las siguientes secciones: ['ANTECEDENTES', 'BASE LEGAL', 'ANÁLISIS', 'CONCLUSIONES', 'RECOMENDACIONES']


In [ ]:
print("EOS Token:", tokenizer.eos_token)
print("EOS Token ID:", tokenizer.eos_token_id)

EOS Token: </s>
EOS Token ID: 2


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/tokenizer.model',
 'lora_model/added_tokens.json',
 'lora_model/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# alpaca_prompt = You MUST copy from above!

inputs = tokenizer(
[
    alpaca_prompt.format(
        "What is a famous tall tower in Paris?", # instruction
        "", # input
        "", # output - leave this blank for generation!
    ),
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

['<s> Redacta un informe legal siguiendo estrictamente la estructura:\n\n1. ANTECEDENTES\n2. BASE LEGAL\n3. ANÁLISIS\n4. CONCLUSIONES\n5. RECOMENDACIONES\n\nAsegúrate de **no terminar el texto sin incluir todas las secciones**.\nFinaliza siempre con RECOMENDACIONES.\n\n### Instruction:\nWhat is a famous tall tower in Paris?\n\n### Input:\n\n\n### Response:\nThe Eiffel Tower is a famous tall tower in Paris.\n\n### Explanation:\nThe Eiffel Tower is a famous tall tower in Paris. It is a wrought iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel']

You can also use Hugging Face's `AutoModelForPeftCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("lora_model")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False:
    model.save_pretrained("model")
    tokenizer.save_pretrained("model")
if False:
    model.push_to_hub("hf/model", token = "")
    tokenizer.push_to_hub("hf/model", token = "")


### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q5_k_m", token = "")

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
